# EduVidQA — Session C: Inference on Kaggle 2×T4

This notebook runs Qwen2.5-VL-7B-Instruct with 4-bit quantization on Kaggle's free T4 GPUs.

**Steps:**
1. Install dependencies
2. Load model (4-bit NF4)
3. Run inference on retrieval results
4. Score answers with Clarity/ECT/UPT
5. Save results

In [ ]:
!pip install -q transformers>=4.45.0 bitsandbytes accelerate qwen-vl-utils torch Pillow pydantic huggingface_hub

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Load Model

In [ ]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("Model loaded.")

## 2. Define Helpers

In [ ]:
import time
from qwen_vl_utils import process_vision_info

SYSTEM_PROMPT = (
    "You are an expert AI teaching assistant helping students understand lecture videos.\n"
    "Your answers must be:\n"
    "1. CLEAR — explain every technical term, use simple language, logical flow\n"
    "2. COMPLETE — cover the concept fully using the provided lecture context\n"
    "3. PEDAGOGICAL — use examples, analogies, step-by-step breakdowns where helpful\n"
    "4. ACCURATE — only state facts supported by the lecture content provided\n\n"
    "You have access to the lecture transcript and video frames around the student's question."
)


def build_messages(question: str, transcript: str, frame_paths: list[str]) -> list[dict]:
    """Build multimodal messages for Qwen2.5-VL."""
    user_content = [
        {"type": "text", "text": f"Lecture transcript:\n{transcript}\n"},
    ]
    for fp in frame_paths[:15]:
        user_content.append({"type": "image", "image": f"file://{fp}"})
    user_content.append({"type": "text", "text": f"\nStudent question: {question}"})
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]


def generate(messages, max_new_tokens=4096, temperature=0.3):
    """Run inference and return (answer_text, elapsed_seconds)."""
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt],
        images=image_inputs or None,
        videos=video_inputs or None,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    t0 = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
        )
    elapsed = time.perf_counter() - t0

    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    answer = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return answer, round(elapsed, 2)

## 3. Run Inference (Text-Only Example)

In [ ]:
# Text-only example (no frames needed)
question = "How does backpropagation work?"
transcript = (
    "Backpropagation works by computing gradients of the loss function "
    "with respect to each weight using the chain rule. The gradient is "
    "then used to update weights in the opposite direction of the gradient descent. "
    "This process repeats for every training example in the dataset."
)

messages = build_messages(question, transcript, frame_paths=[])
answer, elapsed = generate(messages)

print(f"⏱ Generated in {elapsed}s")
print(f"📝 Answer ({len(answer)} chars):\n")
print(answer)

## 4. Quality Scoring

In [ ]:
import json, re
from huggingface_hub import InferenceClient

EVAL_PROMPT = (
    "Rate the following answer on a scale of 1-5 for each quality:\n\n"
    "Question: {question}\n"
    "Answer: {answer}\n\n"
    "Rate EACH on 1-5:\n"
    "1. Clarity (1=jargon-filled → 5=crystal clear)\n"
    "2. ECT - Encouraging Critical Thinking (1=purely factual → 5=poses follow-ups)\n"
    "3. UPT - Uses Pedagogical Techniques (1=no examples → 5=rich examples)\n\n"
    'Return JSON only: {{"clarity": X, "ect": X, "upt": X}}'
)

client = InferenceClient(model="Qwen/Qwen2.5-72B-Instruct")

eval_text = EVAL_PROMPT.format(question=question, answer=answer)
raw = client.text_generation(eval_text, max_new_tokens=100, temperature=0.1)

match = re.search(r"\{[^}]+\}", raw)
if match:
    scores = json.loads(match.group())
    print(f"Clarity: {scores['clarity']}/5")
    print(f"ECT:     {scores['ect']}/5")
    print(f"UPT:     {scores['upt']}/5")
else:
    print("Could not parse scores:", raw)

## 5. Batch Inference (from JSON results file)

In [ ]:
import json
from pathlib import Path

# Load retrieval results (produced by Session B)
RESULTS_PATH = Path("retrieval_results.json")  # Update path as needed

if RESULTS_PATH.exists():
    with open(RESULTS_PATH) as f:
        retrieval_items = json.load(f)

    outputs = []
    for item in retrieval_items:
        transcript = " ".join(ctx["transcript_text"] for ctx in item["contexts"])
        frames = [fp for ctx in item["contexts"] for fp in ctx.get("frame_paths", [])]
        msgs = build_messages(item["query"], transcript, frames)
        ans, elapsed = generate(msgs)
        outputs.append({
            "question": item["query"],
            "answer": ans,
            "video_id": item["video_id"],
            "generation_time_seconds": elapsed,
        })
        print(f"✓ {item['query'][:60]}... ({elapsed}s)")

    with open("answer_results.json", "w") as f:
        json.dump(outputs, f, indent=2)
    print(f"\nSaved {len(outputs)} answers to answer_results.json")
else:
    print(f"{RESULTS_PATH} not found — skipping batch inference.")

In [ ]:
# Cleanup
del model, processor
torch.cuda.empty_cache()
print("Memory freed.")